In [ ]:
import numpy as np
from scipy.signal import welch
from scipy.integrate import simpson
from scipy.stats import entropy, skew, kurtosis, gmean
from scipy.signal import find_peaks
import scipy.io
import pandas as pd
from glob import glob
import os
import glob
from glob import glob
import scipy.io
import numpy as np
from scipy.stats import entropy
import antropy as ant
from scipy.stats import gaussian_kde
#n=250/160 "sampling rate depending on the dataset"

In [ ]:
def compute_psd(signal, fs=n):
    freqs, psd = welch(signal, fs=fs, nperseg=256)
    return freqs, psd

def bandpower(psd, freqs, band, fs=250):
    band_freqs = np.logical_and(freqs >= band[0], freqs <= band[1])
    return simpson(psd[band_freqs], dx=freqs[1] - freqs[0])

def total_power(psd, freqs, fs=250):
    return simpson(psd, dx=freqs[1] - freqs[0])

def relative_power(bp, total_power):
    return bp / total_power

def mean_frequency(psd, freqs):
    return np.sum(freqs * psd) / np.sum(psd)

def median_frequency(psd, freqs):
    cumulative_sum = np.cumsum(psd)
    return freqs[np.searchsorted(cumulative_sum, cumulative_sum[-1] / 2)]

def peak_frequency(psd, freqs):
    return freqs[np.argmax(psd)]

def mu_beta_ratio(psd, freqs):
    mu_power = bandpower(psd, freqs, [9, 13])
    beta_power = bandpower(psd, freqs, [13, 30])
    return mu_power / beta_power

def spectral_entropy(psd):
    return entropy(psd / np.sum(psd))

def spectral_crest_factor(psd):
    return np.max(psd) / np.mean(psd)

def spectral_crest_factor(psd):
    return np.max(psd) / np.mean(psd)

def spectral_skewness(psd, freqs):
    spectral_centroid = np.sum(freqs * psd) / np.sum(psd)
    std_dev = np.sqrt(np.sum(((freqs - spectral_centroid) ** 2) * psd) / np.sum(psd))
    return np.sum((((freqs - spectral_centroid) / std_dev) ** 3) * psd) / np.sum(psd)

def spectral_kurtosis(psd, freqs):
    spectral_centroid = np.sum(freqs * psd) / np.sum(psd)
    std_dev = np.sqrt(np.sum(((freqs - spectral_centroid) ** 2) * psd) / np.sum(psd))
    return np.sum((((freqs - spectral_centroid) / std_dev) ** 4) * psd) / np.sum(psd)

def spectral_flatness(psd):
    geometric_mean = gmean(psd)
    arithmetic_mean = np.mean(psd)
    return geometric_mean / arithmetic_mean


In [ ]:
def compute_features(eeg_data, fs=250):
    n_epochs, n_samples, n_channels = eeg_data.shape
    features = []

    bands = {
        'delta': [0.5, 4],
        'theta': [4, 8],
        'alpha': [8, 13],
        'beta': [13, 30],
        'gamma': [30, 40]
    }

    for epoch in range(n_epochs):
        epoch_features = []
        for channel in range(n_channels):
            signal = eeg_data[epoch, :, channel]
            freqs, psd = compute_psd(signal, fs)

            # Feature list per channel
            channel_features = []

            total_power_value = total_power(psd, freqs)
            for band in bands.values():
                bp = bandpower(psd, freqs, band)
                rp = relative_power(bp, total_power_value)
                channel_features.extend([bp, rp])  # 5 bands × 2 = 10

            mean_freq = mean_frequency(psd, freqs)
            median_freq = median_frequency(psd, freqs)
            peak_freq = peak_frequency(psd, freqs)
            mu_beta = mu_beta_ratio(psd, freqs)
            spec_entropy = spectral_entropy(psd)
            crest_factor = spectral_crest_factor(psd)
            skewness = spectral_skewness(psd, freqs)
            kurt = spectral_kurtosis(psd, freqs)
            flatness = spectral_flatness(psd)

            
            channel_features.extend([
                total_power_value, mean_freq, median_freq, peak_freq,
                mu_beta, spec_entropy, crest_factor, skewness, kurt, flatness
            ])  

            epoch_features.append(channel_features)

        features.append(epoch_features)

    return np.array(features)  # Shape: (n_epochs, n_channels, 20)


In [ ]:
#TIME DOMAIN FEATURES

In [ ]:
import numpy as np
from scipy import stats
import antropy as ant

import numpy as np
from scipy import stats
import antropy as ant


def quartile(data):
    Q1 = np.percentile(data, 25, method='midpoint', axis=-1)
    Q3 = np.percentile(data, 75, method='midpoint', axis=-1)
    IQR = Q3 - Q1
    return IQR


def teager_energy_operator(data):
    def single_teo(signal):
        return np.mean(signal[1:-1]**2 - signal[:-2] * signal[2:])
    return np.apply_along_axis(single_teo, arr=data, axis=-1)


def zero_crossing_rate(data):
    zero_crossings = np.diff(np.sign(data), axis=-1)
    zcr = np.sum(zero_crossings != 0, axis=-1) / (data.shape[-1] - 1)
    return zcr


def ptp(data):
    return np.ptp(data, axis=-1)


def std_dev(data):
    return np.std(data, axis=-1)


def rms(data):
    return np.sqrt(np.mean(data**2, axis=-1))


def skewness(data):
    return stats.skew(data, axis=-1)


def kurtosis(data):
    return stats.kurtosis(data, axis=-1)


def hjorth_mobility(data):
    diff_data = np.diff(data, axis=-1)
    y1 = np.var(diff_data, axis=-1)
    y2 = np.var(data, axis=-1)
    mob = y1 / y2
    mobility = np.sqrt(mob)
    return mobility


def hjorth_complexity(data):
    diff_dat1 = np.diff(data, axis=-1)
    diff_dat2 = np.diff(diff_dat1, axis=-1)
    var1 = np.var(diff_dat2, axis=-1)
    var2 = np.var(diff_dat1, axis=-1)
    div1 = var1 / var2
    mobility1 = np.sqrt(div1)
    var3 = np.var(data, axis=-1)
    div2 = var2 / var3
    mobility2 = np.sqrt(div2)
    complexity = mobility1 / mobility2
    return complexity


def maximum(data):
    return np.max(data, axis=-1)


def mean_absolute_deviation(data):
    g = np.mean(data, axis=-1, keepdims=True)
    mad = np.mean(np.abs(data - g), axis=-1)
    return mad


def energy(data):
    squared_amplitude = np.square(data)
    energy = np.sum(squared_amplitude, axis=-1)
    return energy


def shape_factor(data):
    peak_value = np.max(data, axis=-1)
    rms_value = np.sqrt(np.mean(np.square(data), axis=-1))
    shape_factor_value = peak_value / rms_value
    return shape_factor_value


def clearance_factor(data):
    square_roots = np.sqrt(np.abs(data))
    mean_square_roots = np.mean(square_roots, axis=-1)
    peak_value = np.max(data, axis=-1)
    clearance_factor_value = peak_value / (mean_square_roots ** 2)
    return clearance_factor_value


def AAC(data):
    amplitudes = np.abs(data)
    amplitude_changes = np.diff(amplitudes, axis=-1)
    avg_amplitude_change = np.mean(amplitude_changes, axis=-1)
    return avg_amplitude_change


def katz_fd(data):
    n = data.shape[-1]
    L = np.sum(np.abs(np.diff(data, axis=-1)), axis=-1)
    d = np.max(np.abs(data - data[:, :, 0][:, :, np.newaxis]), axis=-1)
    return np.log10(n) / (np.log10(d / L) + np.log10(n))


def shannon_entropy(data):
    num_samples, num_channels, num_points = data.shape
    entropy_values = []

    for sample in data:
        sample_entropies = []
        for channel in sample:
            # Flatten the channel data
            flattened_data = channel.ravel()

            # Discretize the data into bins (you can adjust the number of bins)
            num_bins = 10
            hist, _ = np.histogram(flattened_data, bins=num_bins)

            # Normalize the histogram to obtain probabilities
            probabilities = hist / len(flattened_data)

            # Calculate Shannon entropy
            entropy_value = stats.entropy(probabilities, base=2)
            sample_entropies.append(entropy_value)

        entropy_values.append(sample_entropies)

    return np.array(entropy_values)

def calculate_hfd(data, kmax=10):
    num_samples, num_channels, num_points = data.shape
    hfd_values = np.zeros((num_samples, num_channels))

    for i in range(num_samples):      # Iterate over trials
        for j in range(num_channels):  # Iterate over channels
            hfd_values[i, j] = ant.higuchi_fd(data[i, j, :].astype(np.float64), kmax=kmax)

    return hfd_values


def hurst_exponent(data):
    N = len(data)
    T = np.arange(1, N + 1)
    Y = np.cumsum(data - np.mean(data))
    
    R = np.maximum.accumulate(Y) - np.minimum.accumulate(Y)
    S = np.std(data)
    if S == 0:
        return 0.5  

    R_S = R / S
    epsilon = 1e-10
    R_S = R_S + epsilon
    H = np.polyfit(np.log(T), np.log(R_S), 1)[0]
    
    return H

def calculate_hurst_matrix(data):
    hurst_matrix = np.zeros((data.shape[0], data.shape[1]))
    for epoch in range(data.shape[0]):
        for ch in range(data.shape[1]):
            hurst_matrix[epoch, ch] = hurst_exponent(data[epoch, ch, :])
    return hurst_matrix

def extract_features(data):
    features = []

    features.append(quartile(data))
    features.append(teager_energy_operator(data))
    features.append(zero_crossing_rate(data))
    features.append(ptp(data))
    features.append(std_dev(data))
    features.append(rms(data))
    features.append(skewness(data))
    features.append(kurtosis(data))
    features.append(hjorth_mobility(data))
    features.append(hjorth_complexity(data))
    features.append(maximum(data))
    features.append(mean_absolute_deviation(data))
    features.append(energy(data))
    features.append(shape_factor(data))
    features.append(clearance_factor(data))
    features.append(AAC(data))
    features.append(katz_fd(data))
    features.append(shannon_entropy(data))
    features.append(calculate_hfd(data))
    features.append(calculate_hurst_matrix(data))

    
    stacked_features = np.stack(features, axis=-1)

    return stacked_features



In [ ]:
features = extract_features(d1)#LOAD INPUT DATA 
print("Extracted features shape:", features.shape)